# Store Data - Simple Baseline

- [Forecasting with ML and Stuff](https://www.kaggle.com/code/ryanholbrook/forecasting-with-machine-learning/tutorial)
- [Kaggle Page](https://www.kaggle.com/competitions/store-sales-time-series-forecasting/data)


In [ ]:
from typing import cast

import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots
from torch import Tensor
from torch.utils.data import DataLoader
from tqdm import tqdm

from time_series.store_sales import MSLELoss, StoreData
from time_series.store_sales_viz import plot_series

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

# Notes

From the information
- Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
- A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

# Load and Explore Data

In [ ]:
store_data = StoreData()
store_data.train

# Model Design

- Input data is essentially `date x store_nbr x family`, which is easily tensor-able. The overall data will be `batch, date_len, store_nbr, family`. 
  - Remember to use `batch_first=True` for any LSTM or whatever models. Apparently having the batch dimension be second is an old convention.
- We need to predict 2017-08-16 to 2017-08-31, so 16 day prediction window
- Input window length will be a parameter, sent to the dataset and model
- loss is Root-Mean-Squared-Logarithmic-Error (given by problem)
  - $\text{RMSE}(\log(1+\hat{y}), \log(1+y))$

# Set Up dataset

## Verify Pandas -> Tensor

In [ ]:
input_pivot = store_data.train.pivot(
    columns=("store_nbr", "family"), values="sales"
).sort_index(axis="columns")

store_data.sales_tensor.shape, store_data.families

In [ ]:
store_id = 5
store_idx = store_id - 1
family = "PREPARED FOODS"
family_idx = cast(int, store_data.families.get_loc(family))

pd_data = input_pivot[(store_id, family)]
pt_data = pd.Series(
    store_data.sales_tensor[:, store_idx, family_idx].numpy(),
    index=input_pivot.index,
)

# Verify data transform is correct
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
)
fig.update_layout(title=f"{store_id=}, '{family}'")

fig.add_trace(
    go.Scatter(
        x=pd_data.index,
        y=pd_data.values,
        name="Input",
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=pt_data.index,
        y=pt_data.values,
        name="Tensor",
    ),
    row=1,
    col=1,
)
error = pd_data - pt_data
fig.add_trace(
    go.Scatter(
        x=error.index,
        y=error.values,
        name="Error",
    ),
    row=2,
    col=1,
)

# Simple Baseline

## Hold Last Sample

In [ ]:
def hold_last_value(input_X: Tensor, input_y: Tensor) -> Tensor:
    match input_X.ndim:
        case 3:
            last_value = input_X[-1:, :, :]
        case 4:
            last_value = input_X[:, -1:, :, :]
        case _:
            raise ValueError(f"{input_X.ndim=}")
    return last_value.expand_as(input_y)

In [ ]:
store_data_loader = DataLoader(store_data, batch_size=10)
loss_func = MSLELoss()

progress_bar = tqdm(store_data_loader)
batch_loss = 0.0
for batch_X, batch_y in progress_bar:
    batch_estimate = hold_last_value(batch_X, batch_y)
    batch_loss += loss_func(batch_estimate, batch_y)
batch_loss

In [ ]:
data_X, data_y = store_data[103]
estimate_y = hold_last_value(data_X, data_y)

plot_series(
    targets=data_y,
    stores=store_data.stores,
    families=store_data.families,
    store_nbr=[3, 4, 5, 6],
    family=store_data.families[7],
    predictions=estimate_y,
)